# Transformers

Build a small Transformer block from PyTorch layers. Trace token IDs through embeddings, position vectors, attention, a feed-forward step, and an output head. The goal is to inspect shapes and masks, not claim trained performance.

In [1]:
import torch
from torch import nn

torch.manual_seed(27)
torch.set_num_threads(1)
print('CPU | small Transformer block')

CPU | small Transformer block


In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, width=8, heads=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(width)
        self.attention = nn.MultiheadAttention(width, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(width)
        self.feed_forward = nn.Sequential(nn.Linear(width, 16), nn.GELU(), nn.Linear(16, width))

    def forward(self, x, mask=None):
        normalized = self.norm1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        return x + self.feed_forward(self.norm2(x))

vocabulary_size, max_length, width = 12, 6, 8
embedding = nn.Embedding(vocabulary_size, width)
position = nn.Embedding(max_length, width)
block = TransformerBlock()
head = nn.Linear(width, vocabulary_size)
token_ids = torch.tensor([[1, 5, 3, 2]])
positions = torch.arange(token_ids.shape[1]).unsqueeze(0)

## Execute the block and verify causal visibility

This forward pass connects the defined modules. The full training, checkpoint selection, held-out evaluation, and generation project is in [lesson 32](32_GPT_and_Autoregressive_Models.ipynb).

In [3]:
block.eval()
causal_mask = torch.triu(torch.ones(token_ids.shape[1], token_ids.shape[1], dtype=torch.bool), diagonal=1)
with torch.no_grad():
    vectors = embedding(token_ids) + position(positions)
    hidden = block(vectors, causal_mask)
    logits = head(hidden)
    changed_ids = token_ids.clone()
    changed_ids[0, -1] = 7
    changed_vectors = embedding(changed_ids) + position(positions)
    changed_hidden = block(changed_vectors, causal_mask)
    torch.testing.assert_close(hidden[:, :-1], changed_hidden[:, :-1], rtol=1e-5, atol=1e-6)
assert vectors.shape == hidden.shape == (1, 4, 8)
assert logits.shape == (1, 4, 12) and torch.isfinite(logits).all()
print('embedded:', tuple(vectors.shape), '| hidden:', tuple(hidden.shape), '| logits:', tuple(logits.shape))
print('Future-token perturbation leaves earlier causal outputs unchanged.')

embedded: (1, 4, 8) | hidden: (1, 4, 8) | logits: (1, 4, 12)
Future-token perturbation leaves earlier causal outputs unchanged.
